<a href="https://colab.research.google.com/github/SoumyaMajumder90/Data-Analysis/blob/main/t5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Question Answering using T5 text to text transfer tranformer.
Using SQuad dataset to finetuning


> Add blockquote



In [ ]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 19.1 MB/s eta 0:00:00


In [ ]:
from transformers import T5ForConditionalGeneration, T5Tokenizer
from datasets import load_dataset
import torch


In [ ]:

model_name = "t5-small"
model = T5ForConditionalGeneration.from_pretrained(model_name)
tokenizer = T5Tokenizer.from_pretrained(model_name)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [ ]:
def answer_question_t5(question, context):

    input_text = f"question: {question}  context: {context}"


    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512)  #  input gone through tokenizer


    outputs = model.generate(inputs.input_ids, max_length=150, num_beams=2)


    answer = tokenizer.decode(outputs[0], skip_special_tokens=True) # decoding output
    return answer


In [ ]:
#context givben
context = """
My name is Soumya Majumder.
"""
question = "What is my name?"


answer = answer_question_t5(question, context)
print("Answer:", answer)


Answer: Soumya Majumder


In [ ]:

squad_dataset = load_dataset("squad")


README.md:   0%|          | 0.00/7.62k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

In [ ]:
# Preprocessing function
def preprocess_function(examples):
    inputs = ["question: " + q + " context: " + c for q, c in zip(examples["question"], examples["context"])]
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding="max_length", return_tensors="pt")
    targets = [a["text"][0] for a in examples["answers"]]
    labels = tokenizer(targets, max_length=128, truncation=True, padding="max_length", return_tensors="pt").input_ids

    # Return batched tensors directly (no splitting)
    return {
        "input_ids": model_inputs["input_ids"].tolist(),  # Convert to list for map compatibility
        "attention_mask": model_inputs["attention_mask"].tolist(),
        "labels": labels.tolist(),
    }

# Preprocess dataset
tokenized_datasets = squad_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=squad_dataset["train"].column_names
)

# Debug: Check type before set_format
print("\nBefore set_format:")
sample = tokenized_datasets["train"][0]
print(f"Sample type: {type(sample)}")
print(f"sample['input_ids'] type: {type(sample['input_ids'])}, shape: {sample['input_ids'].shape if hasattr(sample['input_ids'], 'shape') else 'N/A'}")
print(f"sample['labels'] type: {type(sample['labels'])}, shape: {sample['labels'].shape if hasattr(sample['labels'], 'shape') else 'N/A'}")

# Set format to PyTorch
tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# Debug: Check type after set_format
print("\nAfter set_format:")
sample = tokenized_datasets["train"][0]
print(f"Sample type: {type(sample)}")
print(f"sample['input_ids'] type: {type(sample['input_ids'])}, shape: {sample['input_ids'].shape}")
print(f"sample['labels'] type: {type(sample['labels'])}, shape: {sample['labels'].shape}")

Map:   0%|          | 0/87599 [00:00<?, ? examples/s]

Map:   0%|          | 0/10570 [00:00<?, ? examples/s]


Before set_format:
Sample type: <class 'dict'>
sample['input_ids'] type: <class 'list'>, shape: N/A
sample['labels'] type: <class 'list'>, shape: N/A

After set_format:
Sample type: <class 'dict'>
sample['input_ids'] type: <class 'torch.Tensor'>, shape: torch.Size([512])
sample['labels'] type: <class 'torch.Tensor'>, shape: torch.Size([128])


In [ ]:
from torch.utils.data import DataLoader
train_dataset = tokenized_datasets["train"]
train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)

#checking one batch from dataloader
print("\nFrom DataLoader:")
for batch in train_dataloader:
    print(f"batch type: {type(batch)}")
    print(f"batch['input_ids'] type: {type(batch['input_ids'])}, shape: {batch['input_ids'].shape}")
    print(f"batch['labels'] type: {type(batch['labels'])}, shape: {batch['labels'].shape}")
    break


From DataLoader:
batch type: <class 'dict'>
batch['input_ids'] type: <class 'torch.Tensor'>, shape: torch.Size([16, 512])
batch['labels'] type: <class 'torch.Tensor'>, shape: torch.Size([16, 128])


In [ ]:
#setting up device and which optimizer to use
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

In [ ]:
import torch

# checking if cuda gpu is availablee
print("CUDA available:", torch.cuda.is_available())

# Print Gpu name. print its name
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("Current Device:", torch.cuda.current_device())
else:
    print("Running on CPU")

CUDA available: True
GPU Name: Tesla T4
Current Device: 0


In [ ]:
#Training
num_epochs = 3
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for batch_idx, batch in enumerate(train_dataloader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        if batch_idx % 500 == 0:
            print(f"Epoch {epoch+1}/{num_epochs}, Batch {batch_idx}, Loss: {loss.item():.4f}")

    avg_train_loss = total_loss / len(train_dataloader)
    print(f"Epoch {epoch+1}/{num_epochs}, Average Training Loss: {avg_train_loss:.4f}")

Epoch 1/3, Batch 0, Loss: 0.3015
Epoch 1/3, Batch 500, Loss: 0.0465
Epoch 1/3, Batch 1000, Loss: 0.0367
Epoch 1/3, Batch 1500, Loss: 0.0237
Epoch 1/3, Batch 2000, Loss: 0.0312
Epoch 1/3, Batch 2500, Loss: 0.0251
Epoch 1/3, Batch 3000, Loss: 0.0204
Epoch 1/3, Batch 3500, Loss: 0.0170
Epoch 1/3, Batch 4000, Loss: 0.0158
Epoch 1/3, Batch 4500, Loss: 0.0167
Epoch 1/3, Batch 5000, Loss: 0.0085
Epoch 1/3, Average Training Loss: 0.0331
Epoch 2/3, Batch 0, Loss: 0.0106
Epoch 2/3, Batch 500, Loss: 0.0275
Epoch 2/3, Batch 1000, Loss: 0.0136
Epoch 2/3, Batch 1500, Loss: 0.0173
Epoch 2/3, Batch 2000, Loss: 0.0150
Epoch 2/3, Batch 2500, Loss: 0.0150
Epoch 2/3, Batch 3000, Loss: 0.0200
Epoch 2/3, Batch 3500, Loss: 0.0193
Epoch 2/3, Batch 4000, Loss: 0.0054
Epoch 2/3, Batch 4500, Loss: 0.0153
Epoch 2/3, Batch 5000, Loss: 0.0075
Epoch 2/3, Average Training Loss: 0.0178
Epoch 3/3, Batch 0, Loss: 0.0240
Epoch 3/3, Batch 500, Loss: 0.0211
Epoch 3/3, Batch 1000, Loss: 0.0174
Epoch 3/3, Batch 1500, Loss: 0

In [ ]:
# Save
model.save_pretrained("fine_tuned_t5")
tokenizer.save_pretrained("fine_tuned_t5")

('fine_tuned_t5/tokenizer_config.json',
 'fine_tuned_t5/special_tokens_map.json',
 'fine_tuned_t5/spiece.model',
 'fine_tuned_t5/added_tokens.json')

In [ ]:
!zip -r fine_tuned_t5.zip fine_tuned_t5

  adding: fine_tuned_t5/ (stored 0%)
  adding: fine_tuned_t5/generation_config.json (deflated 29%)
  adding: fine_tuned_t5/added_tokens.json (deflated 83%)
  adding: fine_tuned_t5/special_tokens_map.json (deflated 85%)
  adding: fine_tuned_t5/tokenizer_config.json (deflated 94%)
  adding: fine_tuned_t5/model.safetensors (deflated 9%)
  adding: fine_tuned_t5/spiece.model (deflated 48%)
  adding: fine_tuned_t5/config.json (deflated 62%)


In [ ]:
from google.colab import files
files.download("fine_tuned_t5.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
print("Files in fine_tuned_t5:", os.listdir("fine_tuned_t5"))

Files in fine_tuned_t5: ['generation_config.json', 'added_tokens.json', 'special_tokens_map.json', 'tokenizer_config.json', 'model.safetensors', 'spiece.model', 'config.json']
